# Embeddings for LLMs: Main Code and Experiments

This notebook contains the main code for working with embeddings in large language models (LLMs), plus experiments to understand the impact of the `max_length` and `stride` parameters.

## 1. Import Necessary Libraries

In [10]:
import os
import requests
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.10.0
tiktoken version: 0.12.0


## 2. Load Text Data

We will load the text of "The Verdict" by Edith Wharton, a public domain short story.

In [11]:
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"
    
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

# Read the text
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print(f"Total characters: {len(raw_text)}")
print(f"First 200 characters:\n{raw_text[:200]}")

Total characters: 20479
First 200 characters:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


## 3. Tokenization with BPE (Byte Pair Encoding)

We use the GPT-2 tokenizer through tiktoken.

In [12]:
# Initialize GPT-2 tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Tokenization example
text_example = "Hello, do you like tea? <|endoftext|> In the sunlit terraces."
tokens = tokenizer.encode(text_example, allowed_special={"<|endoftext|>"})

print(f"Original text: {text_example}")
print(f"Tokens: {tokens}")
print(f"Decoded: {tokenizer.decode(tokens)}")

# Tokenize the entire text
enc_text = tokenizer.encode(raw_text)
print(f"\nTotal tokens in the text: {len(enc_text)}")

Original text: Hello, do you like tea? <|endoftext|> In the sunlit terraces.
Tokens: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 13]
Decoded: Hello, do you like tea? <|endoftext|> In the sunlit terraces.

Total tokens in the text: 5145


## 4. Dataset and DataLoader with Sliding Window

We implement a dataset class that uses a sliding window to create samples with `max_length` and `stride`.

In [13]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        # Tokenize the complete text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        
        # Use sliding window to create sequences
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    # Initialize tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")
    
    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    
    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    
    return dataloader

## 5. Token Embeddings and Positional Embeddings

We create embeddings for the tokens and positional embeddings.

In [14]:
# Embeddings configuration
vocab_size = 50257  # GPT-2 vocabulary size
output_dim = 256     # Embeddings dimension
max_length = 4       # Maximum sequence length

# Create embedding layers
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(max_length, output_dim)

# Example with the dataloader
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs shape:", inputs.shape)
print("Inputs:\n", inputs)

# Create token embeddings
token_embeddings = token_embedding_layer(inputs)
print(f"\nToken embeddings shape: {token_embeddings.shape}")

# Create positional embeddings
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(f"Positional embeddings shape: {pos_embeddings.shape}")

# Combine token and positional embeddings
input_embeddings = token_embeddings + pos_embeddings
print(f"Final input embeddings shape: {input_embeddings.shape}")

Inputs shape: torch.Size([8, 4])
Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Token embeddings shape: torch.Size([8, 4, 256])
Positional embeddings shape: torch.Size([4, 256])
Final input embeddings shape: torch.Size([8, 4, 256])


---

# EXPERIMENTS: Impact of max_length and stride

In this section we will experiment with different values of `max_length` and `stride` to understand:
1. **How many samples we get** with different configurations
2. **Why overlap is useful** in training

## Experiment 1: Varying max_length (with stride = max_length)

First, let's see how the number of samples changes when we vary `max_length` while keeping `stride = max_length` (without overlap).

In [15]:
# Experiment 1: Varying max_length
max_lengths = [4, 8, 16, 32, 64, 128]
results_exp1 = []

print("=" * 70)
print("EXPERIMENT 1: Varying max_length (stride = max_length)")
print("=" * 70)

for ml in max_lengths:
    dataset = GPTDatasetV1(raw_text, tokenizer, max_length=ml, stride=ml)
    num_samples = len(dataset)
    results_exp1.append({
        'max_length': ml,
        'stride': ml,
        'num_samples': num_samples,
        'overlap_percent': 0
    })
    print(f"max_length={ml:3d}, stride={ml:3d} → {num_samples:4d} samples (without overlap)")

print("=" * 70)

EXPERIMENT 1: Varying max_length (stride = max_length)
max_length=  4, stride=  4 → 1286 samples (without overlap)
max_length=  8, stride=  8 →  643 samples (without overlap)
max_length= 16, stride= 16 →  321 samples (without overlap)
max_length= 32, stride= 32 →  160 samples (without overlap)
max_length= 64, stride= 64 →   80 samples (without overlap)
max_length=128, stride=128 →   40 samples (without overlap)


## Experiment 2: Varying stride (with fixed max_length)

Now we keep `max_length` fixed and vary `stride` to see how overlap affects the number of samples.

In [16]:
# Experiment 2: Varying stride with fixed max_length
max_length_fixed = 16
strides = [1, 2, 4, 8, 16]  # smaller stride = more overlap
results_exp2 = []

print("=" * 70)
print(f"EXPERIMENT 2: Varying stride (max_length={max_length_fixed})")
print("=" * 70)

for stride in strides:
    dataset = GPTDatasetV1(raw_text, tokenizer, max_length=max_length_fixed, stride=stride)
    num_samples = len(dataset)
    overlap_tokens = max_length_fixed - stride
    overlap_percent = (overlap_tokens / max_length_fixed) * 100
    
    results_exp2.append({
        'max_length': max_length_fixed,
        'stride': stride,
        'num_samples': num_samples,
        'overlap_tokens': overlap_tokens,
        'overlap_percent': overlap_percent
    })
    
    print(f"stride={stride:2d} → {num_samples:4d} samples | "
          f"Overlap: {overlap_tokens:2d} tokens ({overlap_percent:5.1f}%)")

print("=" * 70)

EXPERIMENT 2: Varying stride (max_length=16)
stride= 1 → 5129 samples | Overlap: 15 tokens ( 93.8%)
stride= 2 → 2565 samples | Overlap: 14 tokens ( 87.5%)
stride= 4 → 1283 samples | Overlap: 12 tokens ( 75.0%)
stride= 8 →  642 samples | Overlap:  8 tokens ( 50.0%)
stride=16 →  321 samples | Overlap:  0 tokens (  0.0%)


## Why is Overlap Useful?

Overlap between samples is important because:

1. **Preserves context**: Words at the end of one window maintain connection with those at the beginning of the next
2. **More training data**: We generate more samples from the same text
3. **Better generalization**: The model sees the same text in different contexts
4. **Avoids information loss**: Window boundaries don't cut important information

In [17]:
# Practical demonstration of overlap
print("=" * 70)
print("PRACTICAL DEMONSTRATION: Comparison with and without Overlap")
print("=" * 70)

# Shorter sample text for visualization
sample_text = " ".join(raw_text.split()[:50])  # First 50 words
sample_tokens = tokenizer.encode(sample_text)

max_len = 10

print(f"\nSample text ({len(sample_tokens)} tokens):")
print(f"{sample_text[:200]}...\n")

# Without overlap (stride = max_length)
print(" WITHOUT OVERLAP (stride = max_length = 10):")
print("-" * 70)
dataset_no_overlap = GPTDatasetV1(sample_text, tokenizer, max_length=max_len, stride=max_len)
print(f"Number of samples: {len(dataset_no_overlap)}\n")

for i in range(min(3, len(dataset_no_overlap))):
    tokens = dataset_no_overlap[i][0].tolist()
    text = tokenizer.decode(tokens)
    print(f"Sample {i+1}: {text}")

# With overlap (stride = max_length // 2)
print("\n" + "=" * 70)
print(f" WITH OVERLAP (stride = {max_len//2}, max_length = 10):")
print("-" * 70)
dataset_with_overlap = GPTDatasetV1(sample_text, tokenizer, max_length=max_len, stride=max_len//2)
print(f"Number of samples: {len(dataset_with_overlap)}\n")

for i in range(min(5, len(dataset_with_overlap))):
    tokens = dataset_with_overlap[i][0].tolist()
    text = tokenizer.decode(tokens)
    print(f"Sample {i+1}: {text}")

print("\n" + "=" * 70)
print(f" RESULT: With overlap we obtained {len(dataset_with_overlap)} samples")
print(f"   vs {len(dataset_no_overlap)} without overlap")
print(f"   Increase: {((len(dataset_with_overlap)/len(dataset_no_overlap))-1)*100:.1f}%")
print("=" * 70)

PRACTICAL DEMONSTRATION: Comparison with and without Overlap

Sample text (65 tokens):
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a...

 WITHOUT OVERLAP (stride = max_length = 10):
----------------------------------------------------------------------
Number of samples: 6

Sample 1: I HAD always thought Jack Gisburn rather
Sample 2:  a cheap genius--though a good fellow enough--
Sample 3: so it was no great surprise to me to hear

 WITH OVERLAP (stride = 5, max_length = 10):
----------------------------------------------------------------------
Number of samples: 11

Sample 1: I HAD always thought Jack Gisburn rather
Sample 2:  Jack Gisburn rather a cheap genius--though
Sample 3:  a cheap genius--though a good fellow enough--
Sample 4:  a good fellow enough--so it was no great
Sample 5: so it was no great surprise to me to hear

 RE

## Summary and Conclusions

### Key Findings:

**1. Relationship between max_length and number of samples:**
   - Higher `max_length` : Fewer samples
   - Each window covers more content, we need fewer windows

**2. Relationship between stride and number of samples:**
   - Lower `stride` : More samples and more overlap
   - Higher `stride` : Fewer samples and less overlap
   - `stride = max_length` → No overlap (contiguous windows)

**3. Advantages of Overlap:**
   - More training samples from the same corpus
   - Better context preservation between windows
   - The model sees words in multiple contexts
   - Reduces the risk of losing important information at boundaries

**4. Trade-offs:**
   - More overlap = More samples = More training time
   - Too much overlap can lead to overfitting
   - Typical balance: `stride = max_length / 2` (50% overlap)

## major steps for LLM/agentic systems

## data loading

Data loading is the first step because a language model does not automatically have access to your information. It cannot see your files, your database, or the contents of a website unless you provide that data to it. This step involves taking external information and converting it into text that the system can process. In agentic systems, this is essential because the agent needs real context to answer questions or make decisions. Without loaded data, there is no retrieval, and without retrieval, you cannot build a RAG system. data loading is what transforms external knowledge into something the model can actually use.

## chunking/text splitting
Splitting text into smaller pieces is important because language models have limits on how much information they can handle at once. If you give them a very long document, they may lose precision or fail to process it effectively. Breaking the text into smaller chunks makes it easier to find the exact part that is relevant when someone asks a question. In agentic systems, this greatly improves the quality of responses because it reduces noise and ensures that only the necessary information is retrieved. It also makes the system more efficient and faster. Without chunking, retrieval would be less accurate and the model would struggle to generate clear answers.

## creating embeddings
Creating embeddings is the step where text is converted into numerical representations that capture its meaning. This is essential because machines do not understand words the way humans do; they operate using numbers. Embeddings allow the system to compare texts based on meaning rather than exact word matches. Thanks to embeddings, the model can find related information even if it is written differently. This is what enables semantic search and makes RAG possible. Without embeddings, the system would not be able to determine which pieces of information are relevant to a question, and intelligent retrieval would not work.

## Why do embeddings encode meaning?
Embeddings encode meaning because they are learned from patterns in language. During training, a neural network is exposed to massive amounts of text and learns to predict words based on their context. In doing so, it gradually discovers which words and phrases tend to appear in similar situations. Words that are used in similar contexts end up with similar numerical representations. This means that the position of a word in the embedding space reflects how it is used and what it relates to. As a result, embeddings capture relationships like similarity, category, or even analogies. They do not “understand” meaning the way humans do, but they organize language in a way that reflects how meaning works in practice.

## How are embeddings related to neural network (NN) concepts?
Embeddings are directly connected to how neural networks process information. Inside a neural network, text is transformed through multiple layers of mathematical operations. At some point in this process, the model creates an internal numerical representation of the input — this is essentially the embedding. You can think of it as the model’s internal summary of the text. These representations are learned by adjusting millions or billions of parameters during training. In other words, embeddings are not manually designed; they emerge from the neural network learning patterns in data. They are part of the model’s internal “thinking space,” where meaning is represented as positions in a high-dimensional numerical structure.